# 🔬 Análise Exploratória de Dados (EDA) - FATEC Analytics
**Foco:** Validação Estatística da Cohort de 54 Questões (2010.1 - 2025.1)

O objetivo deste *notebook* é isolar a variância e o desvio padrão da distribuição de gabaritos da FATEC. Responderemos matematicamente a uma crítica comum em modelos preditivos de exames anuais: **"O espaço amostral (Small Data) invalida a tese de Assimetria Controlada?"**

A premissa aqui é provar que, em sistemas engessados por regras de negócio rígidas (como um gabarito que obrigatoriamente deve somar 54 questões), a distribuição não é aleatória (Random Walk), mas sim orquestrada.

In [ ]:
import pandas as pd
import numpy as np

# 1. Recriando o Dataset Histórico da Matriz de 54 Questões
# Cada lista representa o volume histórico de respostas corretas daquela letra.
dados_historicos = {
    'A': [9, 7, 11, 10, 11, 11, 9, 7, 9, 11, 11, 9, 10, 7, 9, 8, 11, 8, 11, 9],
    'B': [12, 11, 11, 12, 10, 12, 12, 9, 11, 9, 12, 8, 11, 15, 12, 15, 5, 5, 11, 11],
    'C': [10, 11, 14, 12, 10, 9, 10, 14, 10, 13, 10, 17, 13, 10, 10, 9, 9, 10, 11, 10],
    'D': [9, 11, 10, 9, 13, 12, 15, 17, 10, 12, 12, 12, 11, 8, 11, 12, 17, 17, 11, 12],
    'E': [14, 14, 8, 11, 10, 10, 8, 7, 14, 9, 9, 8, 9, 14, 12, 10, 12, 14, 10, 12]
}

semestres = [
    '2010.1', '2010.2', '2011.2', '2012.1', '2012.2', '2013.1', '2013.2', 
    '2014.1', '2014.2', '2015.1', '2015.2', '2016.1', '2016.2', '2017.1', 
    '2017.2', '2018.1', '2024.1', '2024.2', '2025.1_A', '2025.1_B'
]

# Construção do DataFrame principal
df_fatec = pd.DataFrame(dados_historicos, index=semestres)

# Validação de Integridade (Sanity Check): Todas as linhas devem somar 54
df_fatec['Total'] = df_fatec.sum(axis=1)
display(df_fatec.head())

### 📊 Teste de Volatilidade e Dispersão (Desvio Padrão)
Se o algoritmo da banca fosse perfeitamente aleatório, o desvio padrão de todas as alternativas seria estatisticamente idêntico. Vamos calcular a média (`mean`), o mínimo (`min`), o máximo (`max`) e a volatilidade (`std`) de cada alternativa para validar nossa tese de "Comportamento de Risco".

In [ ]:
# Removemos a coluna 'Total' para não poluir as métricas
df_metricas = df_fatec.drop(columns=['Total'])

# Geramos a descrição estatística e transpomos para facilitar a leitura (.T)
estatisticas_descritivas = df_metricas.describe().T

# Selecionamos apenas as colunas vitais para a nossa tese
estatisticas_vitais = estatisticas_descritivas[['mean', 'std', 'min', 'max']]

# Ordenamos pelo Desvio Padrão (std) para ver quem é a âncora (menor) e quem é o caos (maior)
estatisticas_vitais = estatisticas_vitais.sort_values(by='std')

print("=== Raio-X Estatístico das Alternativas (54Q) ===")
display(estatisticas_vitais)

### 🧠 Conclusão da Análise de Variância e Defesa do "Small Data"

Os dados provam empiricamente a tese exposta no README:
1. **A Letra A é a âncora de segurança absoluta:** Com o menor desvio padrão global, ela oscila de forma minúscula (banda de 7 a 11). É a opção de preenchimento estrutural da banca.
2. **A Letra B como Armadilha (Honeypot):** Apresenta alta variância de forma bimodal. Manteve-se forte por anos, mas apresenta o menor `min` absoluto da tabela (5 ocorrências), provando o "Crash Point" punitivo do algoritmo.
3. **Letra D como Gatilho de Caos:** Possui um dos maiores desvios padrões e o maior pico de máxima (`max` = 17). 

**E sobre o espaço amostral reduzido (N=20)?** Na estatística clássica, `N < 30` é considerado "Small Data", o que normalmente exigiria a Distribuição T de Student em vez da Curva Normal (Z). Contudo, em engenharia reversa de concursos, **não estamos medindo um fenômeno natural orgânico** (como a altura de uma população). Estamos medindo as regras de restrição de um software (ou design humano) onde os graus de liberdade são artificialmente limitados ($\sum X = 54$). Portanto, as âncoras de variação observadas não são anomalias estatísticas, mas sim a assinatura estrutural inevitável de quem elabora a prova.